In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Pascal Siakam,Over,25.5,-137,2025-11-20,2025-11-19T19:36:44Z
1,PrizePicks,player_points,Pascal Siakam,Under,25.5,-137,2025-11-20,2025-11-19T19:36:44Z
2,PrizePicks,player_points,LaMelo Ball,Over,22.5,-137,2025-11-20,2025-11-19T19:36:44Z
3,PrizePicks,player_points,LaMelo Ball,Under,22.5,-137,2025-11-20,2025-11-19T19:36:44Z
4,PrizePicks,player_points,Miles Bridges,Over,22.5,-137,2025-11-20,2025-11-19T19:36:44Z


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 18 teams with confirmed lineups


### Top EVs for single bets

In [5]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 109 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,KELLY_FRACTION,SIGMA FLAG
0,Josh Giddey,BetRivers,21.5,25.27,Over,120,0,5.06,0.421,High
1,Pelle Larsson,BetRivers,10.5,13.81,Over,112,0,4.92,0.439,High
2,Aaron Gordon,BetRivers,19.5,22.79,Over,120,0,4.69,0.391,High
3,Josh Giddey,BetRivers,20.5,25.27,Over,102,1,4.51,0.442,High
4,Davion Mitchell,BetRivers,10.5,13.07,Over,115,0,4.44,0.386,High


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 77 players...
Processing 70 players with valid predictions...
Generated 2296 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 104 combinations from 2296 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Pelle Larsson,Aaron Gordon,9.5,17.5,13.81,22.79,over,over,0,5.59,0.279,High,High
1,Pelle Larsson,Dereck Lively II,9.5,4.5,13.81,7.09,over,over,0,5.58,0.279,High,Low
2,Aaron Gordon,Dereck Lively II,17.5,4.5,22.79,7.09,over,over,0,5.55,0.277,High,Low
3,Aaron Gordon,Landry Shamet,17.5,9.5,22.79,13.84,over,over,0,5.46,0.273,High,High
4,Dereck Lively II,Landry Shamet,4.5,9.5,7.09,13.84,over,over,0,5.42,0.271,Low,High
5,Pelle Larsson,Landry Shamet,9.5,9.5,13.81,13.84,over,over,0,5.23,0.261,High,High
6,Tony Bradley,Josh Giddey,4.5,20.5,6.89,25.27,over,over,0,4.83,0.241,Low,High
7,Tony Bradley,Kyshawn George,4.5,13.5,6.89,17.97,over,over,0,4.72,0.236,Low,High
8,Tony Bradley,Davion Mitchell,4.5,9.5,6.89,13.07,over,over,0,4.60,0.230,Low,High
9,Davion Mitchell,Josh Giddey,9.5,20.5,13.07,25.27,over,over,0,4.27,0.213,High,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 105 players...
Processing 97 players with valid predictions...
Generated 4427 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 145 combinations from 4427 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Dereck Lively II,Josh Giddey,4.5,19.5,7.09,25.27,over,over,0,5.81,0.290,Low,High
1,Aaron Gordon,Dereck Lively II,17.5,4.5,22.79,7.09,over,over,0,5.54,0.277,High,Low
2,Pelle Larsson,Landry Shamet,9.5,9.5,13.81,13.84,over,over,0,5.50,0.275,High,High
3,Pelle Larsson,Josh Giddey,9.5,19.5,13.81,25.27,over,over,0,5.48,0.274,High,High
4,Aaron Gordon,Landry Shamet,17.5,9.5,22.79,13.84,over,over,0,5.46,0.273,High,High


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 77 players...
Processing 70 players with valid predictions...
Generated 54186 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 46 combinations from 54186 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Aaron Gordon,Dereck Lively II,Landry Shamet,17.5,4.5,9.5,22.79,7.09,13.84,over,over,over,0,11.21,0.224,High,Low,High
1,Pelle Larsson,Aaron Gordon,Dereck Lively II,9.5,17.5,4.5,13.81,22.79,7.09,over,over,over,0,11.10,0.222,High,High,Low
2,Tony Bradley,Pelle Larsson,Landry Shamet,4.5,9.5,9.5,6.89,13.81,13.84,over,over,over,0,10.57,0.211,Low,High,High
3,Tony Bradley,Kyshawn George,Josh Giddey,4.5,13.5,20.5,6.89,17.97,25.27,over,over,over,0,9.15,0.183,Low,High,High
4,Davion Mitchell,Kyshawn George,Josh Giddey,9.5,13.5,20.5,13.07,17.97,25.27,over,over,over,0,8.52,0.170,High,High,High


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 105 players...
Processing 97 players with valid predictions...
Generated 145825 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 64 combinations from 145825 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Aaron Gordon,Dereck Lively II,Josh Giddey,17.5,4.5,19.5,22.79,7.09,25.27,over,over,over,0,11.49,0.230,High,Low,High
1,Landry Shamet,Dereck Lively II,Josh Giddey,9.5,4.5,19.5,13.84,7.09,25.27,over,over,over,0,11.38,0.228,High,Low,High
2,Pelle Larsson,Aaron Gordon,Landry Shamet,9.5,17.5,9.5,13.81,22.79,13.84,over,over,over,0,10.92,0.218,High,High,High
3,Tony Bradley,Pelle Larsson,Kyshawn George,4.5,9.5,13.5,6.89,13.81,17.97,over,over,over,0,9.84,0.197,Low,High,High
4,Tony Bradley,Davion Mitchell,Kyshawn George,4.5,9.5,13.5,6.89,13.07,17.97,over,over,over,0,9.07,0.181,Low,High,High
